# Lesson 1.3 — Subagent Invocation, Context Passing & Spawning

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Use the Task tool to spawn subagents and require allowedTools to include 'Task'
- Pass complete findings into a subagent's prompt since context is not inherited
- Configure AgentDefinition (description, system prompt, tool restrictions) per subagent type
- Spawn parallel subagents by emitting multiple Task calls in a single coordinator response

**You will produce:** An AgentDefinition set plus a context-passing payload that preserves source attribution.

## Concepts

The **Task tool** is the Agent SDK mechanism for spawning subagents. A coordinator can only invoke subagents if its `allowedTools` includes `"Task"`.

- **No inherited context:** subagent context must be *explicitly provided in the prompt*. Subagents do not inherit parent context or share memory between invocations. Pass the actual web results / document analysis directly into the synthesis subagent's prompt.
- **AgentDefinition:** each subagent type is configured with a `description`, a `system prompt`, and `tool restrictions` (which tools it may use).
- **Separate content from metadata:** use structured formats so source URLs, document names, and page numbers survive the handoff and preserve attribution.
- **Parallelism:** spawn parallel subagents by emitting multiple Task tool calls **in a single coordinator response**, not across separate turns.
- **fork_session:** create independent branches from a shared analysis baseline to explore divergent approaches (revisited in Lesson 1.6).

> Verify exact Agent SDK field names/signatures at https://docs.anthropic.com - the SDK evolves; this lesson models the concepts in plain Python.

> **Reference:** [Anthropic - Agent SDK subagents](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Define AgentDefinitions with scoped tools

Each subagent gets a focused description, system prompt, and a restricted tool list.

In [ ]:
AGENT_DEFS = {
    "web_search": {
        "description": "Searches the web and returns sources for a subtopic.",
        "system_prompt": "You find authoritative sources. Return claim + URL + date.",
        "allowedTools": ["WebSearch"],
    },
    "synthesis": {
        "description": "Synthesizes findings into a cited report.",
        "system_prompt": "Combine findings. Preserve every source attribution.",
        "allowedTools": ["verify_fact"],  # scoped cross-role tool (Lesson 2.3)
    },
}

COORDINATOR_TOOLS = ["Task"]  # REQUIRED to spawn subagents
assert "Task" in COORDINATOR_TOOLS
print("Coordinator can spawn subagents:", "Task" in COORDINATOR_TOOLS)

## Code-Along: Step 2 — Build an explicit context payload (content vs metadata)

Because context is not inherited, we hand the synthesis subagent a structured payload that keeps each claim bound to its source - not a blob of prose.

In [ ]:
findings = [
    {"claim": "AI tools cut first-draft time ~40%",
     "evidence": "survey of 1,200 writers",
     "source_url": "https://example.com/writers-study",
     "published": "2026-03-01"},
    {"claim": "Generative music raised licensing disputes",
     "evidence": "industry report excerpt",
     "source_url": "https://example.com/music-report",
     "published": "2026-01-15"},
]

def synthesis_prompt(findings):
    payload = json.dumps(findings, indent=2)
    return (
        "Synthesize these findings into a cited summary. "
        "Preserve claim->source mapping for EVERY claim.\n\nFINDINGS:\n" + payload
    )

print(synthesis_prompt(findings)[:400])

## Code-Along: Step 3 — Spawn parallel subagents in one response

Parallelism comes from emitting multiple Task calls in the *same* coordinator turn. Here we model two Task calls dispatched together.

In [ ]:
def make_task_call(subagent, prompt):
    return {"type": "tool_use", "name": "Task",
            "input": {"subagent_type": subagent, "prompt": prompt}}

# Both emitted in a single assistant response => parallel execution
parallel_turn = [
    make_task_call("web_search", "Find sources on AI in film production (2025-2026)."),
    make_task_call("web_search", "Find sources on AI in music composition (2025-2026)."),
]
print(f"Dispatched {len(parallel_turn)} Task calls in one turn (parallel).")

## Exercise

Add a `doc_analysis` AgentDefinition restricted to a `load_document` tool. Build its prompt from a findings payload that includes `page_number`. Then write an assertion that fails if any finding passed to synthesis is missing a `source_url` - proving attribution survives the handoff.

In [ ]:
# TODO: add AGENT_DEFS['doc_analysis'] with allowedTools=['load_document']
# TODO: build a findings list including page_number metadata
# TODO: assert all('source_url' in f for f in findings)

## Key Takeaways

- The Task tool spawns subagents; the coordinator's allowedTools must include 'Task'.
- Subagents do not inherit context - put complete findings directly in their prompt.
- AgentDefinition sets description, system prompt, and per-agent tool restrictions.
- Separate content from metadata so source URLs/dates survive handoffs.
- Emit multiple Task calls in ONE response to run subagents in parallel.

---

**Next:** [Lesson 1.4 — Multi-Step Workflows: Enforcement & Handoff](./lesson-1.4-enforcement-handoff.ipynb)